In [ ]:
!git clone https://github.com/natasha/slovnet.git
!cd slovnet

!pip install -r requirements/dev.txt
!pip install -e .
!pip install navec razdel pandas tqdm scikit-learn

In [ ]:
TRAIN_CSV = "/kaggle/input/datasets/shashlovaelizaveta/4-data/train_v1 - train_final_1243_v2.csv (11).csv"

TEST_CSV = "/kaggle/input/datasets/shashlovaelizaveta/4-data/test_v1 - test_final_300_v3.csv (20).csv"

NO_PD_CSV = "/kaggle/input/datasets/shashlovaelizaveta/4-data/no_pll_labeled - no_pll_labeled.csv (7).csv"

TRICKY_CSV = "/kaggle/input/datasets/shashlovaelizaveta/4-data/tricky_139_final_labeled - tricky_139_final_labeled_converted.csv (7).csv"

In [ ]:
import ast
import json
import gzip
import pandas as pd
from pathlib import Path

TRAIN_CSV = "/kaggle/input/datasets/shashlovaelizaveta/4-data/train_v1 - train_final_1243_v2.csv (11).csv"



OUT_DIR = Path("/kaggle/working/slovnet/scripts/05_ner/data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_JL_GZ = OUT_DIR / "pii_train.jl.gz"

TARGET_TAGS = {"PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"}


def parse_labels(value):
    if pd.isna(value) or value in ["", "[]"]:
        return []

    if isinstance(value, str):
        value = ast.literal_eval(value)

    spans = []
    for item in value:
        start, stop, entity_type = item
        if entity_type in TARGET_TAGS:
            spans.append({
                "start": int(start),
                "stop": int(stop),
                "type": str(entity_type)
            })
    return spans


df = pd.read_csv(TRAIN_CSV)

with gzip.open(OUT_JL_GZ, "wt", encoding="utf8") as f:
    for _, row in df.iterrows():
        text = str(row["text"])
        spans = parse_labels(row["label"])

        item = {
            "text": text,
            "spans": spans
        }

        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Texts:", len(df))

In [ ]:
import gzip
import json

with gzip.open("/kaggle/working/slovnet/scripts/05_ner/data/pii_train.jl.gz", "rt", encoding="utf8") as f:
    for _ in range(3):
        print(json.loads(next(f)))

In [ ]:
%cd /kaggle/working/slovnet

In [ ]:
!pip install -r requirements/dev.txt

In [ ]:
!pip install -e .

In [ ]:
%cd /kaggle/working/slovnet/scripts/05_ner

In [ ]:
!ls

In [ ]:
!pip install nerus navec slovnet

In [ ]:
!pip install boto3

In [ ]:
!mkdir -p /kaggle/working/slovnet/scripts/05_ner/navec

In [ ]:
!wget https://storage.yandexcloud.net/natasha-navec/packs/navec_news_v1_1B_250K_300d_100q.tar \
-O /kaggle/working/slovnet/scripts/05_ner/navec/navec_news_v1_1B_250K_300d_100q.tar

In [ ]:
!pwd

In [ ]:
!ls /kaggle/working

In [ ]:
!find /kaggle/working -maxdepth 3 -type d | head -50

In [ ]:
!ls /kaggle/working/slovnet/slovnet/model

In [ ]:
%cd /kaggle/working/slovnet/scripts/05_ner

In [ ]:
!mkdir -p data navec model

In [ ]:
!ls data

In [ ]:
import sys
from pathlib import Path

ROOT = Path("/kaggle/working/slovnet")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(sys.path[:3])

In [ ]:
import slovnet
print(slovnet.__file__)

In [ ]:
import slovnet
import slovnet.s3


In [ ]:
import sys, os
from pathlib import Path

ROOT = "/kaggle/working/slovnet"

for name in list(sys.modules):
    if name == "slovnet" or name.startswith("slovnet."):
        del sys.modules[name]

sys.path = [
    p for p in sys.path
    if "slovnet" not in str(p)
]

sys.path.insert(0, ROOT)

print("exists:", Path(ROOT, "slovnet", "__init__.py").exists())
print("s3 exists:", Path(ROOT, "slovnet", "s3.py").exists())
print("sys.path[:5]:", sys.path[:5])



In [ ]:
import slovnet
print("slovnet file:", slovnet.__file__)

import slovnet.s3

In [ ]:
!ls /kaggle/working/slovnet/slovnet

In [ ]:
!pwd
!ls -lah data

In [ ]:
CUSTOM_TEXTS = "data/pii_train.jl.gz"

In [ ]:
import gzip, json
from collections import Counter

cnt = Counter()

with gzip.open("data/pii_train.jl.gz", "rt", encoding="utf8") as f:
    for line in f:
        item = json.loads(line)
        for span in item.get("spans", []):
            cnt[span["type"]] += 1

cnt

In [ ]:
import sys
from pathlib import Path

%cd /kaggle/working/slovnet/scripts/05_ner
%run main.py
CUSTOM_TEXTS = "/kaggle/working/slovnet/scripts/05_ner/data/pii_train_hard_no_pd.jl.gz"


TAGS = ["PERSON", "ADDRESS", "EMAIL", "ID", "PHONE"]
DEVICE = "cpu"

LAYERS_NUM = 4
LAYER_DIM = 128
LAYER_DIMS = [
    LAYER_DIM * 2 ** _
    for _ in reversed(range(LAYERS_NUM))
]

LR = 0.0005
LR_GAMMA = 0.95
EPOCHS = 15

BATCH_SIZE = 8
SEQ_LEN = 256
SHUFFLE_SIZE = 1000

if not exists(NAVEC):
    !wget {NAVEC_URL} -O {NAVEC}

navec = Navec.load(NAVEC)

words_vocab = Vocab(navec.vocab.words)
shapes_vocab = Vocab([PAD] + SHAPES)
tags_vocab = BIOTagsVocab(TAGS)

torch.manual_seed(SEED)
seed(SEED)

word = NavecEmbedding(navec)

shape = Embedding(
    vocab_size=len(shapes_vocab),
    dim=SHAPE_DIM,
    pad_id=shapes_vocab.pad_id,
)

emb = TagEmbedding(word, shape)

encoder = TagEncoder(
    input_dim=emb.dim,
    layer_dims=LAYER_DIMS,
    kernel_size=KERNEL_SIZE,
)

ner = NERHead(encoder.dim, len(tags_vocab))
model = NER(emb, encoder, ner).to(DEVICE)

lines = load_gz_lines(CUSTOM_TEXTS)
items = parse_jl(lines)

markups = (SpanMarkup.from_json(_) for _ in items)
markups = (_.to_bio(list(tokenize(_.text))) for _ in markups)

encode = TagTrainEncoder(
    words_vocab,
    shapes_vocab,
    tags_vocab,
    seq_len=SEQ_LEN,
    batch_size=BATCH_SIZE,
    shuffle_size=SHUFFLE_SIZE,
)

batches = list(encode(markups))
batches = [_.to(DEVICE) for _ in batches]

size = max(1, int(len(batches) * 0.15))

batches = {
    TEST: batches[:size],
    TRAIN: batches[size:],
}


optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, LR_GAMMA)

meters = {
    TRAIN: NERScoreMeter(),
    TEST: NERScoreMeter(),
}

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    model.train()
    for batch in batches[TRAIN]:
        optimizer.zero_grad()
        batch = process_batch(model, ner.crf, batch)
        batch.loss.backward()
        optimizer.step()
        meters[TRAIN].add(NERBatchScore(batch.loss))

    print("Train:")
    meters[TRAIN].write(LogBoard())
    meters[TRAIN].reset()

    model.eval()
    with torch.no_grad():
        for batch in batches[TEST]:
            batch = process_batch(model, ner.crf, batch)
            batch.pred = ner.crf.decode(batch.pred)
            meters[TEST].add(score_ner_batch(batch, tags_vocab))

    print("Valid:")
    meters[TEST].write(LogBoard())
    meters[TEST].reset()

    scheduler.step()

model.emb.shape.dump(MODEL_SHAPE)
model.encoder.dump(MODEL_ENCODER)
ner.dump(MODEL_NER)

!ls model

In [ ]:
%cd /kaggle/working/slovnet/scripts/05_ner
%run main.py

TAGS = ["PERSON", "ADDRESS", "EMAIL", "ID", "PHONE"]
tags_vocab = BIOTagsVocab(TAGS)

LAYERS_NUM = 4
LAYER_DIM = 128
LAYER_DIMS = [
    LAYER_DIM * 2 ** _
    for _ in reversed(range(LAYERS_NUM))
]

print(LAYER_DIMS)

!mkdir -p navec model

if not exists(NAVEC):
    !wget {NAVEC_URL} -O {NAVEC}

navec = Navec.load(NAVEC)

words_vocab = Vocab(navec.vocab.words)
shapes_vocab = Vocab([PAD] + SHAPES)
tags_vocab = BIOTagsVocab(TAGS)

word = NavecEmbedding(navec)

shape = Embedding(
    vocab_size=len(shapes_vocab),
    dim=SHAPE_DIM,
    pad_id=shapes_vocab.pad_id,
)

emb = TagEmbedding(word, shape)

encoder = TagEncoder(
    input_dim=emb.dim,
    layer_dims=LAYER_DIMS,
    kernel_size=KERNEL_SIZE,
)

ner_head = NERHead(encoder.dim, len(tags_vocab))
pack_model = NER(emb, encoder, ner_head)
pack_model.eval()

pack_model.emb.shape.load(MODEL_SHAPE)
pack_model.encoder.load(MODEL_ENCODER)
pack_model.head.load(MODEL_NER)

pack_model = pack_model.to_exec()
pack_model = pack_model.strip_navec()
arrays, pack_model = pack_model.separate_arrays()

PACK = PACK_NAME

with DumpPack(PACK) as pack:
    meta = Meta(ID)
    pack.dump_meta(meta)
    pack.dump_model(pack_model)
    pack.dump_arrays(arrays)
    pack.dump_vocab(words_vocab, WORD)
    pack.dump_vocab(shapes_vocab, SHAPE)
    pack.dump_vocab(tags_vocab, TAG)


In [ ]:
TEST_CSV = "/kaggle/input/datasets/shashlovaelizaveta/test-sm/test_small - test_final_300_v3.csv.csv"
NO_PD_CSV = "/kaggle/working/no_pd_eval_clean.csv"
TRICKY_CSV = "/kaggle/input/datasets/shashlovaelizaveta/4-data/tricky_139_final_labeled - tricky_139_final_labeled_converted.csv (7).csv"

In [ ]:
import pandas as pd

NO_PD_PATH = "/kaggle/input/datasets/shashlovaelizaveta/4-data/no_pll_labeled - no_pll_labeled.csv (7).csv"

HARD_NO_PD_TRAIN_PATH = "/kaggle/working/train_with_hard_no_pd.csv"

OUT_PATH = "/kaggle/working/no_pd_eval_clean.csv"

no_pd_df = pd.read_csv(NO_PD_PATH)

train_aug_df = pd.read_csv(HARD_NO_PD_TRAIN_PATH)

hard_no_pd_texts = set(
    train_aug_df[
        train_aug_df["source"].astype(str).str.contains("hard_no_pd")
    ]["text"].astype(str)
)


no_pd_eval_df = no_pd_df[
    ~no_pd_df["text"].astype(str).isin(hard_no_pd_texts)
].copy()

no_pd_eval_df.to_csv(OUT_PATH, index=False)



In [ ]:
import ast
import pandas as pd
from navec import Navec
from slovnet import NER as SlovnetNER

TAGS = ["PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"]

NAVEC_PATH = "/kaggle/working/slovnet/scripts/05_ner/navec/navec_news_v1_1B_250K_300d_100q.tar"
PACK_PATH = f"/kaggle/working/slovnet/scripts/05_ner/{PACK_NAME}"

navec = Navec.load(NAVEC_PATH)

ner_model = SlovnetNER.load(PACK_PATH)
ner_model.navec(navec)


def parse_gold_labels(value):
    if pd.isna(value) or value == "" or value == "[]":
        return []

    if isinstance(value, str):
        value = ast.literal_eval(value)

    return [
        (int(start), int(stop), str(label))
        for start, stop, label in value
    ]


def load_labeled_csv(path):
    df = pd.read_csv(path)

    df["gold"] = df["label"].apply(parse_gold_labels)
    return df


def predict_spans(text):
    markup = ner_model(str(text))
    spans = []

    for span in markup.spans:
        label = str(span.type)

        if label in TAGS:
            spans.append((int(span.start), int(span.stop), label))

    return spans


def calc_prf(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    return precision, recall, f1


def evaluate_main_test(df):
    stats = {
        tag: {"TP": 0, "FP": 0, "FN": 0}
        for tag in TAGS
    }

    for _, row in df.iterrows():
        text = row["text"]

        gold = set(row["gold"])
        pred = set(predict_spans(text))

        for tag in TAGS:
            gold_tag = {x for x in gold if x[2] == tag}
            pred_tag = {x for x in pred if x[2] == tag}

            stats[tag]["TP"] += len(gold_tag & pred_tag)
            stats[tag]["FP"] += len(pred_tag - gold_tag)
            stats[tag]["FN"] += len(gold_tag - pred_tag)

    rows = []

    for tag in TAGS:
        tp = stats[tag]["TP"]
        fp = stats[tag]["FP"]
        fn = stats[tag]["FN"]

        precision, recall, f1 = calc_prf(tp, fp, fn)

        rows.append({
            "Класс": tag,
            "Precision": round(precision, 4),
            "Recall": round(recall, 4),
            "F1": round(f1, 4),
            "TP": tp,
            "FP": fp,
            "FN": fn,
        })

    per_class = pd.DataFrame(rows)

    total_tp = sum(stats[tag]["TP"] for tag in TAGS)
    total_fp = sum(stats[tag]["FP"] for tag in TAGS)
    total_fn = sum(stats[tag]["FN"] for tag in TAGS)

    micro_p, micro_r, micro_f1 = calc_prf(total_tp, total_fp, total_fn)

    summary = {
        "micro_precision": round(micro_p, 4),
        "micro_recall": round(micro_r, 4),
        "micro_f1": round(micro_f1, 4),
        "macro_f1": round(per_class["F1"].mean(), 4),
        "mean_f1_PERSON_ADDRESS": round(
            per_class[
                per_class["Класс"].isin(["PERSON", "ADDRESS"])
            ]["F1"].mean(),
            4
        ),
    }

    return per_class, summary


def evaluate_no_pd(df):
    n_texts_with_error = 0
    fp_entities = 0

    for _, row in df.iterrows():
        pred = predict_spans(row["text"])

        if len(pred) > 0:
            n_texts_with_error += 1
            fp_entities += len(pred)

    return {
        "n_texts": len(df),
        "n_texts_with_error": n_texts_with_error,
        "fp_entities": fp_entities,
        "fp_text_rate": round(n_texts_with_error / len(df), 4),
    }


def evaluate_tricky(df):
    total_fp = 0
    total_fn = 0
    texts_with_any_error = 0

    for _, row in df.iterrows():
        gold = set(row["gold"])
        pred = set(predict_spans(row["text"]))

        fp = len(pred - gold)
        fn = len(gold - pred)

        total_fp += fp
        total_fn += fn

        if fp > 0 or fn > 0:
            texts_with_any_error += 1

    return {
        "n_texts": len(df),
        "FP": total_fp,
        "FN": total_fn,
        "texts_with_any_error": texts_with_any_error,
        "error_text_rate": round(texts_with_any_error / len(df), 4),
    }

NO_PD_CSV = "/kaggle/working/no_pd_eval_clean.csv"
test_df = load_labeled_csv(TEST_CSV)
no_pd_df = load_labeled_csv(NO_PD_CSV)
tricky_df = load_labeled_csv(TRICKY_CSV)

test_per_class, test_summary = evaluate_main_test(test_df)
no_pd_metrics = evaluate_no_pd(no_pd_df)
tricky_metrics = evaluate_tricky(tricky_df)

display(test_per_class)

for k, v in test_summary.items():
    print(f"{k}: {v}")

for k, v in no_pd_metrics.items():
    print(f"{k}: {v}")

for k, v in tricky_metrics.items():
    print(f"{k}: {v}")

In [ ]:
for idx, row in test_df.iterrows():

    text = str(row["text"])

    gold = set(row["gold"])
    pred = set(predict_spans(text))

    gold_person = {
        x for x in gold
        if x[2] == "PERSON"
    }

    pred_person = {
        x for x in pred
        if x[2] == "PERSON"
    }

    fp = pred_person - gold_person
    fn = gold_person - pred_person

    if len(fp) == 0 and len(fn) == 0:
        continue

    print(f"TEXT ID: {idx}")
    print()
    print(text)
    print()


    if len(fp) > 0:
        print("FP PERSON:")

        for start, stop, label in fp:
            print(
                f"  -> '{text[start:stop]}' "
                f"[{start}:{stop}]"
            )

    if len(fn) > 0:
        print("FN PERSON:")

        for start, stop, label in fn:
            print(
                f"  -> '{text[start:stop]}' "
                f"[{start}:{stop}]"
            )

    print()

In [ ]:
import ast
import pandas as pd

def parse_gold_labels(value):
    if pd.isna(value) or value == "" or value == "[]":
        return []

    if isinstance(value, str):
        value = ast.literal_eval(value)

    return [
        (int(start), int(stop), str(label))
        for start, stop, label in value
    ]


def span_text(text, start, stop):
    return text[start:stop]

def collect_no_pd_errors(df):
    rows = []

    for idx, row in df.iterrows():

        text = str(row["text"])

        pred = set(predict_spans(text))

        for start, stop, label in pred:

            rows.append({
                "dataset": "NO_PD",
                "text_id": idx,
                "error_type": "FP",
                "label": label,
                "gold": "",
                "predicted": span_text(text, start, stop),
                "start": start,
                "stop": stop,
                "text": text
            })

    return pd.DataFrame(rows)

def collect_tricky_errors(df):

    rows = []

    for idx, row in df.iterrows():

        text = str(row["text"])

        gold = set(row["gold"])
        pred = set(predict_spans(text))

        fp = pred - gold

        for start, stop, label in fp:

            rows.append({
                "dataset": "TRICKY",
                "text_id": idx,
                "error_type": "FP",
                "label": label,
                "gold": "",
                "predicted": span_text(text, start, stop),
                "start": start,
                "stop": stop,
                "text": text
            })

        fn = gold - pred

        for start, stop, label in fn:

            rows.append({
                "dataset": "TRICKY",
                "text_id": idx,
                "error_type": "FN",
                "label": label,
                "gold": span_text(text, start, stop),
                "predicted": "",
                "start": start,
                "stop": stop,
                "text": text
            })

    return pd.DataFrame(rows)


In [ ]:
no_pd_df = load_labeled_csv(NO_PD_CSV)
tricky_df = load_labeled_csv(TRICKY_CSV)

In [ ]:
no_pd_errors_df = collect_no_pd_errors(no_pd_df)

tricky_errors_df = collect_tricky_errors(tricky_df)

In [ ]:
no_pd_error_texts = no_pd_errors_df["text_id"].nunique()
tricky_error_texts = tricky_errors_df["text_id"].nunique()

print("NO_PD texts with errors:", no_pd_error_texts)


In [ ]:
print("TRICKY texts with errors:", tricky_error_texts)

In [ ]:
NO_PD_OUT = "/kaggle/working/no_pd_errors.csv"
TRICKY_OUT = "/kaggle/working/tricky_errors.csv"

no_pd_errors_df.to_csv(NO_PD_OUT, index=False)
tricky_errors_df.to_csv(TRICKY_OUT, index=False)



In [ ]:
print(TRICKY_OUT)

In [ ]:
display(no_pd_errors_df.head(20))


In [ ]:
display(tricky_errors_df.head(20))

**Гибрид. Модель**

Гибрид. Сбор

In [ ]:
!pip install yargy pymorphy2

PHONE

In [ ]:
import ast
import re
import time
import pandas as pd

from yargy import Parser, rule, or_
from yargy.predicates import eq, caseless, custom


ENTITY_CLASSES = ["PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"]


def load_dataset(path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, sep="\t")
        if "text" not in df.columns:
            df = pd.read_csv(path)
    except Exception:
        df = pd.read_csv(path)

    if "label" in df.columns:
        df["label"] = df["label"].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
    else:
        df["label"] = [[] for _ in range(len(df))]

    return df



def normalize_gold(label_list):
    return [
        {"start": int(x[0]), "end": int(x[1]), "label": x[2]}
        for x in label_list
    ]


def entity_key(ent):
    return (ent["start"], ent["end"], ent["label"])


def resolve_overlaps(entities):
    entities = sorted(
        entities,
        key=lambda e: (e["start"], -(e["end"] - e["start"]))
    )

    selected = []

    for ent in entities:
        has_overlap = False

        for old in selected[:]:
            if not (ent["end"] <= old["start"] or ent["start"] >= old["end"]):
                has_overlap = True

                if (ent["end"] - ent["start"]) > (old["end"] - old["start"]):
                    selected.remove(old)
                    selected.append(ent)

                break

        if not has_overlap:
            selected.append(ent)

    return sorted(selected, key=lambda e: e["start"])


def extend_phone_span(text: str, start: int, end: int):


    tail = text[end:end + 50]

    paren_ext = re.match(
        r"\s*\(\s*(?:вн\.?|доб\.?|ext\.?)\s*\d{1,6}\s*\)",
        tail,
        flags=re.IGNORECASE
    )

    ext = re.match(
        r"\s*(?:доб\.?|вн\.?|ext\.?|#)\s*\d{1,6}",
        tail,
        flags=re.IGNORECASE
    )

    if paren_ext:
        end += paren_ext.end()
    elif ext:
        end += ext.end()

    tail = text[end:end + 40]

    list_tail = re.match(
        r"(?:\s*[,/]\s*\d{2}){1,4}",
        tail
    )

    if list_tail:
        end += list_tail.end()

    return start, end


def is_bad_phone_candidate(text: str, start: int, end: int, source: str = "") -> bool:
    value = text[start:end]
    digits = re.sub(r"\D", "", value)
    stripped = value.strip()

    left_context_15 = text[max(0, start - 15):start].lower()
    left_context_40 = text[max(0, start - 40):start].lower()

    phone_context = re.search(
        r"(тел\.?|телефон|моб\.?|номер телефона|контактный телефон|контактный номер|для связи)\s*[:\-]?\s*$",
        left_context_40,
        flags=re.IGNORECASE
    )

    bad_context = re.search(
        r"(номер документа|регистрационный номер|номер обращения|id клиента|user_id|id|инн|р/с|счет|номер счета|идентификатор)\s*[:=]?\s*$",
        left_context_40,
        flags=re.IGNORECASE
    )

    if "в/ч" in left_context_15:
        return True

    if bad_context and not phone_context:
        return True

    if stripped.isdigit() and len(digits) < 10:
        return True

    if stripped.isdigit() and len(digits) >= 11:
        if not re.match(r"^(?:\+?7|8)\d{10}$", stripped) and not phone_context:
            return True

    if re.fullmatch(r"\d{1,3}-\d{2}-\d{2}(?:\s*(?:[,/]\s*\d{2}){1,4})?", stripped):
        if not phone_context:
            return True

    if re.search(
        r"(код|артикул|номер договора|номер сч[её]та|номер заявки|код заказа|код операции)",
        left_context_40,
        flags=re.IGNORECASE
    ):
        if not re.search(
            r"(тел\.?|телефон|моб\.?|для связи)",
            left_context_40,
            flags=re.IGNORECASE
        ):
            return True

    return False


class YargyPhoneDetector:
    def __init__(self):
        def token_value(t):
            return t.value if hasattr(t, "value") else str(t)

        def digits_pred(pattern):
            return custom(lambda t: bool(re.fullmatch(pattern, token_value(t))))

        def token_pred(pattern):
            return custom(
                lambda t: bool(
                    re.fullmatch(
                        pattern,
                        token_value(t),
                        flags=re.IGNORECASE | re.VERBOSE
                    )
                )
            )

        DIG1 = digits_pred(r"\d{1}")
        DIG2 = digits_pred(r"\d{2}")
        DIG3 = digits_pred(r"\d{3}")
        DIG4 = digits_pred(r"\d{4}")
        DIG7 = digits_pred(r"\d{7}")
        DIG10 = digits_pred(r"\d{10}")
        DIG11 = digits_pred(r"\d{11}")
        DIG1_2 = digits_pred(r"\d{1,2}")
        DIG2_3 = digits_pred(r"\d{2,3}")
        DIG2_4 = digits_pred(r"\d{2,4}")
        DIG1_6 = digits_pred(r"\d{1,6}")
        DIG6 = digits_pred(r"\d{6}")

        PHONE_COMPACT_TOKEN = token_pred(
            r"""
            (?:
                \+?7[\.\-\s]?\(?\d{3}\)?[\.\-\s]?\d{3}[\.\-\s]?\d{2}[\.\-\s]?\d{2}
                |
                8[\.\-\s]?\(?\d{3}\)?[\.\-\s]?\d{3}[\.\-\s]?\d{2}[\.\-\s]?\d{2}
                |
                8\(\d{3}\)\d{7}
                |
                \+7\(\d{3}\)\d{7}
                |
                \d{3}-\d{3}-\d{3}\s?\d{2}
                |
                \d{4}-\d{7}
            )
            """
        )

        CODE_PAREN_TOKEN = token_pred(r"\(\d{3}\)")

        SEP = or_(rule(eq("-")), rule(eq(".")))

        PLUS7 = rule(eq("+"), eq("7"))

        PREFIX = or_(
            PLUS7,
            rule(eq("7")),
            rule(eq("8")),
        )

        CODE_PAREN = or_(
            rule(eq("("), DIG3, eq(")")),
            rule(CODE_PAREN_TOKEN),
        )

        CODE = or_(
            rule(DIG3),
            CODE_PAREN,
        )

        PHONE_SINGLE_TOKEN = rule(PHONE_COMPACT_TOKEN)

        PHONE_PLUS_SOLID = rule(eq("+"), DIG11)

        PHONE_BROKEN_PAREN = rule(
            PLUS7,
            eq("("),
            DIG10,
        )

        PHONE_DOT_FULL = rule(
            PLUS7,
            eq("."),
            DIG3,
            eq("."),
            DIG3,
            eq("."),
            DIG2,
            eq("."),
            DIG2,
        )

        PHONE_SOLID_11 = rule(DIG11)

        PHONE_8_3_4_3 = rule(
            eq("8"),
            DIG3,
            DIG4,
            DIG3,
        )

        PHONE_3_3_3_2 = rule(
            DIG3,
            eq("-"),
            DIG3,
            eq("-"),
            DIG3,
            DIG2,
        )

        PHONE_WITH_PREFIX = rule(
            PREFIX,
            SEP.optional(),
            CODE,
            SEP.optional(),
            DIG2_4,
            SEP.optional(),
            DIG1_2,
            SEP.optional(),
            DIG2_3,
        )

        PHONE_PREFIX_CODE_7DIG = rule(
            PREFIX,
            CODE,
            DIG7,
        )

        PHONE_PAREN_NO_PREFIX = rule(
            CODE_PAREN,
            DIG3,
            eq("-"),
            DIG2,
            eq("-"),
            DIG2,
        )

        PHONE_NO_PREFIX_GROUPED = rule(
            DIG3,
            DIG3,
            DIG2,
            DIG2,
        )

        PHONE_WITHOUT_PREFIX = or_(
            rule(DIG10),
            rule(
                CODE,
                SEP.optional(),
                DIG3,
                SEP.optional(),
                DIG2,
                SEP.optional(),
                DIG2,
            ),
        )

        LOCAL_PHONE = or_(
            rule(DIG3, eq("-"), DIG2, eq("-"), DIG2),
            rule(DIG2, eq("-"), DIG2, eq("-"), DIG2),
            rule(DIG1, eq("-"), DIG2, eq("-"), DIG2),
        )

        PHONE_WEIRD = or_(
            rule(DIG4, eq("-"), DIG7),
            rule(eq("8"), DIG3, SEP, DIG7),
            rule(eq("8"), DIG6, DIG2, DIG2),
            rule(PLUS7, DIG10),
        )

        LIST_TAIL = or_(
            rule(eq("/"), DIG2),
            rule(eq(","), DIG2),
            rule(eq(","), DIG2, eq(","), DIG2),
        )

        EXT_MARKER = or_(
            rule(caseless("доб")),
            rule(caseless("вн")),
            rule(caseless("ext")),
            rule(eq("#")),
        )

        EXTENSION = or_(
            rule(EXT_MARKER, eq(".").optional(), DIG1_6),
            rule(eq("("), caseless("вн"), eq(".").optional(), DIG1_6, eq(")")),
        )

        BASE_PHONE = or_(
            PHONE_SINGLE_TOKEN,
            PHONE_PLUS_SOLID,
            PHONE_BROKEN_PAREN,
            PHONE_DOT_FULL,
            PHONE_PREFIX_CODE_7DIG,
            PHONE_8_3_4_3,
            PHONE_3_3_3_2,
            PHONE_PAREN_NO_PREFIX,
            PHONE_NO_PREFIX_GROUPED,
            PHONE_WITH_PREFIX,
            PHONE_WEIRD,
            PHONE_WITHOUT_PREFIX,
            PHONE_SOLID_11,
            LOCAL_PHONE,
        )

        PHONE_WORDS = rule(
            caseless("восемь"),
            caseless("девятьсот"),
            caseless("двадцать"),
            caseless("шесть"),
            caseless("сто"),
            caseless("двадцать"),
            caseless("три"),
            caseless("сорок"),
            caseless("пять"),
            caseless("шестьдесят"),
            caseless("семь"),
        )

        PHONE = or_(
            rule(
                BASE_PHONE,
                LIST_TAIL.optional(),
                EXTENSION.optional(),
            ),
            PHONE_WORDS,
        )

        self.parser = Parser(PHONE)

        self.phone_regexes = [
            r"(?<!\d)(?:\+7|8)\.\d{3}\.\d{3}\.\d{2}\.\d{2}(?!\d)",

            r"(?<!\d)(?:\+7|8)\(\d{3}\)\d{7}(?!\d)",

            r"(?<!\d)(?:\+?7|8)\d{10}(?!\d)",

            r"(?<!\d)(?:\+7|8)[-\s]\d{3}[-\s]\d{3}[-\s]\d{2}[-\s]\d{2}(?!\d)",

            r"(?<!\d)(?:\+7|8)\s*\(\d{3}\)\s*\d{3}[\s-]\d{2}[\s-]\d{2}(?!\d)",

            r"(?<!\d)(?:\+7|8)\s*\(\d{3}\)\s*\d{7}(?!\d)",

            r"(?<!\d)(?:\+7|8|7)\s+\d{3}\s+\d{3}\s+\d{2}\s+\d{2}(?!\d)",

            r"(?<!\d)\d{3}\s+\d{3}\s+\d{2}\s+\d{2}(?!\d)",

            r"(?<!\d)\(\d{3}\)\s*\d{3}-\d{2}-\d{2}(?!\d)",

            r"(?<!\d)\+?7\d{10}(?:\s*(?:доб\.?|вн\.?|ext\.?|#)\s*\d{1,6}|\s*\(\s*(?:вн\.?|доб\.?|ext\.?)\s*\d{1,6}\s*\))?(?!\d)",

            r"(?<!\d)8\s*\(\d{3}\)\s*\d{7}(?:\s*(?:доб\.?|вн\.?|ext\.?|#)\s*\d{1,6}|\s*\(\s*(?:вн\.?|доб\.?|ext\.?)\s*\d{1,6}\s*\))?(?!\d)",

            r"(?<![\d-])\d{3}-\d{2}-\d{2}(?:\s*(?:[,/]\s*\d{2}){1,4})?(?![\d-])",

            r"(?<!\d)\d{1,2}-\d{2}-\d{2}(?!\d)",
        ]

    def _add_entity(self, entities, text, start, end, source):
        if is_bad_phone_candidate(text, start, end, source=source):
            return

        start, end = extend_phone_span(text, start, end)

        entities.append({
            "start": start,
            "end": end,
            "label": "PHONE",
            "text": text[start:end],
            "source": source,
        })

    def predict_one(self, text: str):
        entities = []

        for match in self.parser.findall(text):
            self._add_entity(
                entities=entities,
                text=text,
                start=match.span.start,
                end=match.span.stop,
                source="yargy",
            )

        for pattern in self.phone_regexes:
            for m in re.finditer(pattern, text, flags=re.IGNORECASE):
                self._add_entity(
                    entities=entities,
                    text=text,
                    start=m.start(),
                    end=m.end(),
                    source="regex_phone",
                )

        return resolve_overlaps(entities)


def calculate_metrics(gold_all, pred_all, classes=ENTITY_CLASSES):
    per_class = {}
    total_tp = total_fp = total_fn = 0

    for cls in classes:
        tp = fp = fn = 0

        for gold, pred in zip(gold_all, pred_all):
            gold_cls = {entity_key(e) for e in gold if e["label"] == cls}
            pred_cls = {entity_key(e) for e in pred if e["label"] == cls}

            tp += len(gold_cls & pred_cls)
            fp += len(pred_cls - gold_cls)
            fn += len(gold_cls - pred_cls)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall)
            else 0.0
        )

        per_class[cls] = {
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "TP": tp,
            "FP": fp,
            "FN": fn,
        }

        total_tp += tp
        total_fp += fp
        total_fn += fn

    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if (micro_precision + micro_recall)
        else 0.0
    )

    macro_f1 = sum(per_class[c]["F1"] for c in classes) / len(classes)

    return {
        "per_class": per_class,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
    }


def metrics_to_table(metrics):
    rows = []

    for cls in ENTITY_CLASSES:
        m = metrics["per_class"][cls]
        rows.append({
            "Класс": cls,
            "Precision": round(m["Precision"], 4),
            "Recall": round(m["Recall"], 4),
            "F1": round(m["F1"], 4),
            "TP": m["TP"],
            "FP": m["FP"],
            "FN": m["FN"],
        })

    return pd.DataFrame(rows)


def run_yargy_phone_experiment(path: str):
    df = load_dataset(path)
    detector = YargyPhoneDetector()

    texts = df["text"].tolist()
    gold = [normalize_gold(x) for x in df["label"].tolist()]

    start_time = time.time()
    predictions = [detector.predict_one(text) for text in texts]
    elapsed = time.time() - start_time

    metrics = calculate_metrics(gold, predictions)
    class_table = metrics_to_table(metrics)

    summary = {
        "method": "Yargy-PHONE-plus-regex",
        "micro_precision": round(metrics["micro_precision"], 4),
        "micro_recall": round(metrics["micro_recall"], 4),
        "micro_f1": round(metrics["micro_f1"], 4),
        "macro_f1": round(metrics["macro_f1"], 4),
        "time_sec": round(elapsed, 4),
        "texts_per_sec": round(len(df) / elapsed, 4) if elapsed > 0 else None,
    }

    pred_rows = []
    for i, (text, preds, gold_labels) in enumerate(zip(texts, predictions, gold)):
        pred_rows.append({
            "id": i,
            "text": text,
            "gold": gold_labels,
            "predictions": preds,
        })

    predictions_df = pd.DataFrame(pred_rows)

    print(pd.DataFrame([summary]).to_string(index=False))

    print(class_table.to_string(index=False))

    return summary, class_table, predictions_df

def print_phone_errors(predictions_df: pd.DataFrame):
    rows = []

    for _, row in predictions_df.iterrows():
        gold = [e for e in row["gold"] if e["label"] == "PHONE"]
        pred = [e for e in row["predictions"] if e["label"] == "PHONE"]

        gold_set = {entity_key(e) for e in gold}
        pred_set = {entity_key(e) for e in pred}

        fp = [p for p in pred if entity_key(p) not in gold_set]
        fn = [g for g in gold if entity_key(g) not in pred_set]

        if fp or fn:
            rows.append({
                "id": row["id"],
                "text": row["text"],
                "false_positive": fp,
                "false_negative": fn,
            })

    errors_df = pd.DataFrame(rows)

    print(f"Количество текстов с ошибками: {len(errors_df)}")

    for _, row in errors_df.iterrows():
        print(f"ID: {row['id']}")
        print(f"Текст: {row['text']}")

        if row["false_positive"]:
            print("FP:")
            for e in row["false_positive"]:
                print(f'  "{e["text"]}" [{e["start"]}:{e["end"]}] → PHONE')

        if row["false_negative"]:
            print("FN:")
            for e in row["false_negative"]:
                start, end = e["start"], e["end"]
                print(f'  "{row["text"][start:end]}" [{start}:{end}] → PHONE')

    return errors_df




ID

In [ ]:
import ast
import re
import time
import pandas as pd

from yargy import Parser, rule, or_
from yargy.predicates import eq, caseless, custom


ENTITY_CLASSES = ["PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"]


def load_dataset(path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, sep="\t")
        if "text" not in df.columns:
            df = pd.read_csv(path)
    except Exception:
        df = pd.read_csv(path)

    if "label" in df.columns:
        df["label"] = df["label"].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
    else:
        df["label"] = [[] for _ in range(len(df))]

    return df


def normalize_gold(label_list):
    return [
        {"start": int(x[0]), "end": int(x[1]), "label": x[2]}
        for x in label_list
    ]


def entity_key(ent):
    return (ent["start"], ent["end"], ent["label"])


def resolve_overlaps(entities):
    entities = sorted(
        entities,
        key=lambda e: (e["start"], -(e["end"] - e["start"]))
    )

    selected = []

    for ent in entities:
        has_overlap = False

        for old in selected[:]:
            if not (ent["end"] <= old["start"] or ent["start"] >= old["end"]):
                has_overlap = True

                if (ent["end"] - ent["start"]) > (old["end"] - old["start"]):
                    selected.remove(old)
                    selected.append(ent)

                break

        if not has_overlap:
            selected.append(ent)

    return sorted(selected, key=lambda e: e["start"])


class YargyIdDetector:
    def __init__(self):
        def token_value(t):
            return t.value if hasattr(t, "value") else str(t)

        def digits_pred(pattern):
            return custom(lambda t: bool(re.fullmatch(pattern, token_value(t))))

        def token_pred(pattern):
            return custom(
                lambda t: bool(
                    re.fullmatch(pattern, token_value(t), flags=re.IGNORECASE)
                )
            )

        DIG1_2 = digits_pred(r"\d{1,2}")
        DIG2 = digits_pred(r"\d{2}")
        DIG3 = digits_pred(r"\d{3}")
        DIG4 = digits_pred(r"\d{4}")
        DIG1_20 = digits_pred(r"\d{1,20}")
        DIG11 = digits_pred(r"\d{11}")
        DIG12 = digits_pred(r"\d{12}")

        RU_LETTERS = token_pred(r"[А-ЯЁA-Z]{1,6}")
        ALNUM = token_pred(r"[A-ZА-ЯЁ0-9]{2,20}")

        NUM_SIGN = eq("№")

        ID_MARKER = or_(
            rule(caseless("ID")),
            rule(caseless("id")),
            rule(caseless("user_id")),
        )

        ID_CONTEXT_MARKER = or_(
            rule(caseless("ID"), caseless("клиента")),
            rule(caseless("id"), caseless("клиента")),
            rule(caseless("клиента")),
            rule(caseless("регистрационный"), caseless("ID")),
            rule(caseless("регистрационный"), caseless("id")),
            rule(caseless("идентификатор")),
        )

        DOC_MARKER = or_(
            rule(caseless("договор")),
            rule(caseless("договором")),
            rule(caseless("контракт")),
            rule(caseless("контрактом")),
            rule(caseless("регистрационный"), caseless("контракт")),
            rule(caseless("заявка")),
            rule(caseless("заявки")),
            rule(caseless("заявок")),
            rule(caseless("номер"), caseless("заявки")),
            rule(caseless("номер"), caseless("заявок")),
            rule(caseless("номера"), caseless("заявок")),
            rule(caseless("регистрационные"), caseless("заявки")),
            rule(caseless("идентификаторы"), caseless("заявок")),
            rule(caseless("идентификатор"), caseless("заявки")),
            rule(caseless("документ")),
            rule(caseless("идентификатор")),
            rule(caseless("номер"), caseless("документа")),
            rule(caseless("номер"), caseless("договора")),
            rule(caseless("номер"), caseless("контракта")),
            rule(caseless("номер"), caseless("обращения")),
            rule(caseless("регистрационный"), caseless("номер")),
            rule(caseless("регистрационный"), caseless("номер"), caseless("заявки")),
        )

        INN_MARKER = rule(caseless("ИНН"))
        SNILS_MARKER = rule(caseless("СНИЛС"))

        ACCOUNT_MARKER = or_(
            rule(caseless("счет")),
            rule(caseless("счёт")),
            rule(caseless("номер"), caseless("счета")),
            rule(caseless("номер"), caseless("счёта")),
            rule(caseless("номером"), caseless("счета")),
            rule(caseless("номером"), caseless("счёта")),
            rule(caseless("регистрационный"), caseless("номер"), caseless("счета")),
            rule(caseless("регистрационный"), caseless("номер"), caseless("счёта")),
            rule(caseless("р"), eq("/"), caseless("с")),
        )

        SIMPLE_ID = or_(
            rule(ID_MARKER, eq(":").optional(), eq("=").optional(), DIG1_20),
            rule(ID_CONTEXT_MARKER, eq(":").optional(), NUM_SIGN.optional(), DIG1_20),
            rule(NUM_SIGN, DIG1_20),
        )

        DOC_NUMBER_ALNUM = or_(
            rule(RU_LETTERS, eq("-"), DIG4, eq("/"), DIG1_2, eq("-"), DIG1_2),
            rule(DIG4, eq("-"), DIG1_20),
            rule(DIG3, eq("-"), DIG3, eq("-"), ALNUM),
            rule(DIG3, eq("-"), DIG3, eq("-"), DIG3, DIG2),
            rule(DIG3, eq("-"), DIG3, eq("-"), DIG3, eq("/"), DIG1_2),
            rule(DIG3, eq("-"), DIG3, eq("-"), DIG3, eq("-"), DIG2),
            rule(DIG1_2, eq("-"), DIG3, eq("-"), RU_LETTERS),
            rule(DIG1_20, eq("-"), DIG1_20),
            rule(DIG1_20, eq("-"), RU_LETTERS),
            rule(DIG1_20),
        )

        DOC_ID = or_(
            rule(DOC_MARKER, eq(":").optional(), NUM_SIGN.optional(), DOC_NUMBER_ALNUM),
            rule(NUM_SIGN, DOC_NUMBER_ALNUM),
        )

        INN_ID = rule(INN_MARKER, eq(":").optional(), DIG12)

        SNILS_FORMATTED = rule(
            SNILS_MARKER,
            eq(":").optional(),
            DIG3,
            eq("-"),
            DIG3,
            eq("-"),
            DIG3,
            DIG2,
        )

        SNILS_SOLID_WITH_MARKER = rule(
            SNILS_MARKER,
            eq(":").optional(),
            DIG11,
        )

        GOVERNMENT_ID = or_(
            INN_ID,
            SNILS_FORMATTED,
            SNILS_SOLID_WITH_MARKER,
        )

        BANK_ACCOUNT = rule(
            ACCOUNT_MARKER,
            eq(":").optional(),
            DIG1_20,
        )

        ENUM_WITH_MARKER = or_(
            rule(caseless("заявки"), eq(":").optional(), DIG1_20),
            rule(caseless("заявка"), eq(":").optional(), DIG1_20),
            rule(caseless("заявок"), eq(":").optional(), DIG1_20),
            rule(caseless("номера"), caseless("заявок"), eq(":").optional(), DIG1_20),
            rule(caseless("номер"), caseless("заявок"), eq(":").optional(), DIG1_20),
            rule(caseless("идентификаторы"), caseless("заявок"), eq(":").optional(), DIG1_20),
            rule(caseless("регистрационные"), caseless("заявки"), eq(":").optional(), DIG1_20),
        )

        ID = or_(
            GOVERNMENT_ID,
            BANK_ACCOUNT,
            DOC_ID,
            SIMPLE_ID,
            ENUM_WITH_MARKER,
        )

        self.parser = Parser(ID)

    def trim_id_span(self, text: str, start: int, end: int):
        value = text[start:end]

        patterns = [
            r"^(?:ID|id|user_id)\s*[:=]?\s*",
            r"^(?:клиента)\s*:?\s*№?\s*",
            r"^(?:ИНН|СНИЛС)\s*:?\s*",
            r"^(?:№)\s*",

            r"^(?:ом)\s*:?\s*№?\s*",

            r"^(?:а)\s+(?=\d)",

            r"^(?:документ)\s*:?\s*№?\s*",
            r"^(?:договор|договором|договора|контракт|контрактом|контракта|регистрационный\s+контракт)\s*:?\s*№?\s*",
            r"^(?:заявка|заявки|заявок|номер\s+заявки|номер\s+заявок|номера\s+заявок|регистрационные\s+заявки|идентификаторы\s+заявок|идентификатор\s+заявки|регистрационный\s+номер\s+заявки)\s*:?\s*№?\s*",
            r"^(?:номер\s+документа|номер\s+договора|номер\s+контракта|номер\s+обращения|регистрационный\s+номер)\s*:?\s*№?\s*",
            r"^(?:ID\s+клиента|id\s+клиента|регистрационный\s+ID|регистрационный\s+id|идентификатор)\s*:?\s*№?\s*",
            r"^(?:счет|счёт|счета|счёта|номером\s+счета|номером\s+счёта|номер\s+счета|номер\s+счёта|регистрационный\s+номер\s+счета|регистрационный\s+номер\s+счёта|р/с)\s*:?\s*",
        ]

        changed = True
        while changed:
            changed = False
            value = text[start:end]

            for pattern in patterns:
                m = re.match(pattern, value, flags=re.IGNORECASE)
                if m:
                    start += m.end()
                    changed = True
                    break

        return start, end

    def extend_id_span(self, text: str, start: int, end: int):
        value = text[start:end]

        if re.fullmatch(r"\d{1,4}", value):
            tail = text[end:end + 40]

            m = re.match(
                r"(?:-\d{1,6})?(?:-[A-ZА-ЯЁ0-9]{1,20})?(?:/\d{1,4})?",
                tail,
                flags=re.IGNORECASE
            )

            if m:
                end += m.end()

        return start, end

    def split_enumerated_ids(self, text: str, start: int, end: int):
        value = text[start:end]

        if re.fullmatch(r"\d{1,20}(?:\s*,\s*\d{1,20})+", value):
            return [
                (start + m.start(), start + m.end())
                for m in re.finditer(r"\d{1,20}", value)
            ]

        return [(start, end)]

    def _is_bad_id_candidate(self, text: str, start: int, end: int) -> bool:
        value = text[start:end]
        value_lower = value.lower()
        digits = re.sub(r"\D", "", value)

        left_context_80 = text[max(0, start - 80):start].lower()
        right_context_30 = text[end:end + 30]

        phone_context = re.search(
            r"(тел\.?|телефон|моб\.?|контактный номер|номер телефона|для связи)\s*[:\-]?\s*$",
            left_context_80,
            flags=re.IGNORECASE
        )

        if phone_context:
            return True

        bad_left_context = re.search(
            r"(школ[аеуыи]?|поликлиник[аеуыи]?|изолятор[аеуыи]?|общежити[ея]|"
            r"дом|кв\.?|квартира|офис|оф\.?|стр\.?|корп\.?|"
            r"рейс|шаг|этап)\s*$",
            left_context_80,
            flags=re.IGNORECASE
        )

        if bad_left_context:
            return True

        if re.fullmatch(r"\d{1,3}-\d{3}", value) and re.match(
            r"-\d{2,3}-\d{2}-\d{2}",
            right_context_30
        ):
            return True

        has_marker = re.search(
            r"(id|user_id|инн|снилс|№|договор|контракт|заявк|сч[её]т|р/с|номер|регистрационный|идентификатор|клиента)",
            value_lower + " " + left_context_80,
            flags=re.IGNORECASE
        )

        if len(digits) <= 3 and not has_marker:
            return True

        return False

    def _add_entity(self, entities, text, start, end, source):
        if self._is_bad_id_candidate(text, start, end):
            return

        start, end = self.trim_id_span(text, start, end)
        start, end = self.trim_id_span(text, start, end)
        start, end = self.extend_id_span(text, start, end)

        if start >= end:
            return

        for sub_start, sub_end in self.split_enumerated_ids(text, start, end):
            if self._is_bad_id_candidate(text, sub_start, sub_end):
                continue

            entities.append({
                "start": sub_start,
                "end": sub_end,
                "label": "ID",
                "text": text[sub_start:sub_end],
                "source": source,
            })

    def predict_one(self, text: str):
        entities = []

        for match in self.parser.findall(text):
            self._add_entity(
                entities,
                text,
                match.span.start,
                match.span.stop,
                "yargy_id",
            )

        for m in re.finditer(
            r"(?:идентификаторы|номера|номер|регистрационные)?\s*заяв(?:ок|ки)\s*:\s*((?:\d{1,20}\s*,\s*)+\d{1,20})",
            text,
            flags=re.IGNORECASE
        ):
            nums_start = m.start(1)
            nums_part = m.group(1)

            for num in re.finditer(r"\d{1,20}", nums_part):
                sub_start = nums_start + num.start()
                sub_end = nums_start + num.end()

                entities.append({
                    "start": sub_start,
                    "end": sub_end,
                    "label": "ID",
                    "text": text[sub_start:sub_end],
                    "source": "yargy_id_enum",
                })

        for m in re.finditer(
            r"(?:идентификаторы|идентификатор)\s*:\s*((?:\d{1,20}\s*,\s*)+\d{1,20})",
            text,
            flags=re.IGNORECASE
        ):
            nums_start = m.start(1)
            nums_part = m.group(1)

            for num in re.finditer(r"\d{1,20}", nums_part):
                sub_start = nums_start + num.start()
                sub_end = nums_start + num.end()

                entities.append({
                    "start": sub_start,
                    "end": sub_end,
                    "label": "ID",
                    "text": text[sub_start:sub_end],
                    "source": "yargy_id_enum",
                })

        return resolve_overlaps(entities)


def calculate_metrics(gold_all, pred_all, classes=ENTITY_CLASSES):
    per_class = {}
    total_tp = total_fp = total_fn = 0

    for cls in classes:
        tp = fp = fn = 0

        for gold, pred in zip(gold_all, pred_all):
            gold_cls = {entity_key(e) for e in gold if e["label"] == cls}
            pred_cls = {entity_key(e) for e in pred if e["label"] == cls}

            tp += len(gold_cls & pred_cls)
            fp += len(pred_cls - gold_cls)
            fn += len(gold_cls - pred_cls)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall)
            else 0.0
        )

        per_class[cls] = {
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "TP": tp,
            "FP": fp,
            "FN": fn,
        }

        total_tp += tp
        total_fp += fp
        total_fn += fn

    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if (micro_precision + micro_recall)
        else 0.0
    )

    macro_f1 = sum(per_class[c]["F1"] for c in classes) / len(classes)

    return {
        "per_class": per_class,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
    }


def metrics_to_table(metrics):
    rows = []

    for cls in ENTITY_CLASSES:
        m = metrics["per_class"][cls]
        rows.append({
            "Класс": cls,
            "Precision": round(m["Precision"], 4),
            "Recall": round(m["Recall"], 4),
            "F1": round(m["F1"], 4),
            "TP": m["TP"],
            "FP": m["FP"],
            "FN": m["FN"],
        })

    return pd.DataFrame(rows)



def run_yargy_id_experiment(path: str):
    df = load_dataset(path)
    detector = YargyIdDetector()

    texts = df["text"].astype(str).tolist()
    gold = [normalize_gold(x) for x in df["label"].tolist()]

    start_time = time.time()
    predictions = [detector.predict_one(text) for text in texts]
    elapsed = time.time() - start_time

    metrics = calculate_metrics(gold, predictions)
    class_table = metrics_to_table(metrics)

    summary = {
        "method": "Yargy-ID-only",
        "micro_precision": round(metrics["micro_precision"], 4),
        "micro_recall": round(metrics["micro_recall"], 4),
        "micro_f1": round(metrics["micro_f1"], 4),
        "macro_f1": round(metrics["macro_f1"], 4),
        "time_sec": round(elapsed, 4),
        "texts_per_sec": round(len(df) / elapsed, 4) if elapsed > 0 else None,
    }

    pred_rows = []
    for i, (text, preds, gold_labels) in enumerate(zip(texts, predictions, gold)):
        pred_rows.append({
            "id": i,
            "text": text,
            "gold": gold_labels,
            "predictions": preds,
        })

    predictions_df = pd.DataFrame(pred_rows)

    print(pd.DataFrame([summary]).to_string(index=False))

    print(class_table.to_string(index=False))

    return summary, class_table, predictions_df


def print_id_errors(predictions_df: pd.DataFrame):
    rows = []

    for _, row in predictions_df.iterrows():
        gold = [e for e in row["gold"] if e["label"] == "ID"]
        pred = [e for e in row["predictions"] if e["label"] == "ID"]

        gold_set = {entity_key(e) for e in gold}
        pred_set = {entity_key(e) for e in pred}

        fp = [p for p in pred if entity_key(p) not in gold_set]
        fn = [g for g in gold if entity_key(g) not in pred_set]

        if fp or fn:
            rows.append({
                "id": row["id"],
                "text": row["text"],
                "false_positive": fp,
                "false_negative": fn,
            })

    errors_df = pd.DataFrame(rows)

    print(f"Количество текстов с ошибками: {len(errors_df)}")

    for _, row in errors_df.iterrows():
        print("\n---")
        print(f"ID: {row['id']}")
        print(f"Текст: {row['text']}")

        if row["false_positive"]:
            print("FP:")
            for e in row["false_positive"]:
                print(f'  "{e["text"]}" [{e["start"]}:{e["end"]}] → ID')

        if row["false_negative"]:
            print("FN:")
            for e in row["false_negative"]:
                start, end = e["start"], e["end"]
                print(f'  "{row["text"][start:end]}" [{start}:{end}] → ID')

    return errors_df




EMAIL

In [ ]:
import ast
import re
import time
import pandas as pd

from yargy import Parser, rule, or_
from yargy.predicates import eq, custom


ENTITY_CLASSES = ["PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"]

def load_dataset(path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, sep="\t")
        if "text" not in df.columns:
            df = pd.read_csv(path)
    except Exception:
        df = pd.read_csv(path)

    if "label" in df.columns:
        df["label"] = df["label"].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
    else:
        df["label"] = [[] for _ in range(len(df))]

    return df



def normalize_gold(label_list):
    return [
        {"start": int(x[0]), "end": int(x[1]), "label": x[2]}
        for x in label_list
    ]


def entity_key(ent):
    return (ent["start"], ent["end"], ent["label"])


def resolve_overlaps(entities):
    entities = sorted(
        entities,
        key=lambda e: (e["start"], -(e["end"] - e["start"]))
    )

    selected = []

    for ent in entities:
        has_overlap = False

        for old in selected[:]:
            if not (ent["end"] <= old["start"] or ent["start"] >= old["end"]):
                has_overlap = True

                if (ent["end"] - ent["start"]) > (old["end"] - old["start"]):
                    selected.remove(old)
                    selected.append(ent)

                break

        if not has_overlap:
            selected.append(ent)

    return sorted(selected, key=lambda e: e["start"])



class YargyEmailDetector:
    def __init__(self):
        def token_value(t):
            return t.value if hasattr(t, "value") else str(t)

        def token_pred(pattern):
            return custom(
                lambda t: bool(
                    re.fullmatch(pattern, token_value(t), flags=re.IGNORECASE)
                )
            )

        LOCAL_PART = token_pred(
            r"[A-ZА-ЯЁ0-9]+(?:[._+\-][A-ZА-ЯЁ0-9]+)*"
        )

        DOMAIN_PART = token_pred(
            r"[A-ZА-ЯЁ0-9]+(?:-[A-ZА-ЯЁ0-9]+)*"
        )

        TLD = token_pred(
            r"[A-ZА-ЯЁ0-9]{2,20}"
        )

        EMAIL_FULL_TOKEN = token_pred(
            r"[A-ZА-ЯЁ0-9]+(?:[._+\-][A-ZА-ЯЁ0-9]+)*"
            r"@"
            r"[A-ZА-ЯЁ0-9]+(?:-[A-ZА-ЯЁ0-9]+)*"
            r"(?:\.[A-ZА-ЯЁ0-9]+(?:-[A-ZА-ЯЁ0-9]+)*|,[A-ZА-ЯЁ]{2,15})+"
        )

        DOT_OR_COMMA = or_(rule(eq(".")), rule(eq(",")))

        EMAIL_BASIC = rule(
            LOCAL_PART,
            eq("@"),
            DOMAIN_PART,
            DOT_OR_COMMA,
            TLD,
        )

        EMAIL_MULTI_DOMAIN = rule(
            LOCAL_PART,
            eq("@"),
            DOMAIN_PART,
            eq("."),
            DOMAIN_PART,
            DOT_OR_COMMA,
            TLD,
        )

        EMAIL_THREE_DOMAIN = rule(
            LOCAL_PART,
            eq("@"),
            DOMAIN_PART,
            eq("."),
            DOMAIN_PART,
            eq("."),
            DOMAIN_PART,
            DOT_OR_COMMA,
            TLD,
        )

        EMAIL = or_(
            rule(EMAIL_FULL_TOKEN),
            EMAIL_THREE_DOMAIN,
            EMAIL_MULTI_DOMAIN,
            EMAIL_BASIC,
        )

        self.parser = Parser(EMAIL)

        self.email_regex = re.compile(
            r"[A-ZА-ЯЁ0-9]+(?:\s*[._+\-]\s*[A-ZА-ЯЁ0-9]+)*"
            r"\s*@\s*"
            r"[A-ZА-ЯЁ0-9]+(?:\s*-\s*[A-ZА-ЯЁ0-9]+)*"
            r"(?:"
            r"\s*\.\s*[A-ZА-ЯЁ0-9]+(?:\s*-\s*[A-ZА-ЯЁ0-9]+)*"
            r"|,[A-ZА-ЯЁ]{2,15}"
            r")+",
            flags=re.IGNORECASE
        )

    def normalize_email_candidate(self, value: str) -> str:
        compact = re.sub(r"\s+", "", value)
        compact = compact.replace(",", ".")
        return compact

    def trim_email_span(self, text: str, start: int, end: int):
        value = text[start:end]

        m = re.search(r",\s+", value)
        if m:
            end = start + m.start()

        while end > start and text[end - 1] in ".,;:!?)]}":
            end -= 1

        return start, end

    def is_bad_email_candidate(self, text: str, start: int, end: int) -> bool:
        start, end = self.trim_email_span(text, start, end)
        value = text[start:end]
        compact = self.normalize_email_candidate(value)

        if start > 0:
            prev_char = text[start - 1]
            if re.match(r"[A-Za-zА-Яа-яЁё0-9._+\-]", prev_char):
                return True

        if end < len(text):
            next_char = text[end]
            if re.match(r"[A-Za-zА-Яа-яЁё0-9_\-]", next_char):
                return True

        if not re.fullmatch(
            r"[A-ZА-ЯЁ0-9]+(?:[._+\-][A-ZА-ЯЁ0-9]+)*"
            r"@"
            r"[A-ZА-ЯЁ0-9]+(?:-[A-ZА-ЯЁ0-9]+)*"
            r"(?:\.[A-ZА-ЯЁ0-9]+(?:-[A-ZА-ЯЁ0-9]+)*)+",
            compact,
            flags=re.IGNORECASE
        ):
            return True

        return False

    def extend_email_span(self, text: str, start: int, end: int):
        window_start = max(0, start - 100)
        window_end = min(len(text), end + 100)
        window = text[window_start:window_end]

        best = None

        for m in self.email_regex.finditer(window):
            candidate_start = window_start + m.start()
            candidate_end = window_start + m.end()
            candidate_start, candidate_end = self.trim_email_span(
                text,
                candidate_start,
                candidate_end
            )

            if candidate_start <= start and candidate_end >= end:
                if best is None or (candidate_end - candidate_start) > (best[1] - best[0]):
                    best = (candidate_start, candidate_end)

        if best:
            return best

        return self.trim_email_span(text, start, end)

    def _add_entity(self, entities, text, start, end, source):
        start, end = self.extend_email_span(text, start, end)
        start, end = self.trim_email_span(text, start, end)

        if self.is_bad_email_candidate(text, start, end):
            return

        entities.append({
            "start": start,
            "end": end,
            "label": "EMAIL",
            "text": text[start:end],
            "source": source,
        })

    def predict_one(self, text: str):
        entities = []

        for match in self.parser.findall(text):
            self._add_entity(
                entities,
                text,
                match.span.start,
                match.span.stop,
                "yargy_email",
            )

        for m in self.email_regex.finditer(text):
            self._add_entity(
                entities,
                text,
                m.start(),
                m.end(),
                "regex_email",
            )

        return resolve_overlaps(entities)



def calculate_metrics(gold_all, pred_all, classes=ENTITY_CLASSES):
    per_class = {}
    total_tp = total_fp = total_fn = 0

    for cls in classes:
        tp = fp = fn = 0

        for gold, pred in zip(gold_all, pred_all):
            gold_cls = {entity_key(e) for e in gold if e["label"] == cls}
            pred_cls = {entity_key(e) for e in pred if e["label"] == cls}

            tp += len(gold_cls & pred_cls)
            fp += len(pred_cls - gold_cls)
            fn += len(gold_cls - pred_cls)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0

        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall)
            else 0.0
        )

        per_class[cls] = {
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "TP": tp,
            "FP": fp,
            "FN": fn,
        }

        total_tp += tp
        total_fp += fp
        total_fn += fn

    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0

    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if (micro_precision + micro_recall)
        else 0.0
    )

    macro_f1 = sum(per_class[c]["F1"] for c in classes) / len(classes)

    return {
        "per_class": per_class,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
    }


def metrics_to_table(metrics):
    rows = []

    for cls in ENTITY_CLASSES:
        m = metrics["per_class"][cls]
        rows.append({
            "Класс": cls,
            "Precision": round(m["Precision"], 4),
            "Recall": round(m["Recall"], 4),
            "F1": round(m["F1"], 4),
            "TP": m["TP"],
            "FP": m["FP"],
            "FN": m["FN"],
        })

    return pd.DataFrame(rows)



def run_yargy_email_experiment(path: str):
    df = load_dataset(path)
    detector = YargyEmailDetector()

    texts = df["text"].astype(str).tolist()
    gold = [normalize_gold(x) for x in df["label"].tolist()]

    start_time = time.time()
    predictions = [detector.predict_one(text) for text in texts]
    elapsed = time.time() - start_time

    metrics = calculate_metrics(gold, predictions)
    class_table = metrics_to_table(metrics)

    summary = {
        "method": "Yargy-EMAIL-only",
        "micro_precision": round(metrics["micro_precision"], 4),
        "micro_recall": round(metrics["micro_recall"], 4),
        "micro_f1": round(metrics["micro_f1"], 4),
        "macro_f1": round(metrics["macro_f1"], 4),
        "time_sec": round(elapsed, 4),
        "texts_per_sec": round(len(df) / elapsed, 4) if elapsed > 0 else None,
    }

    pred_rows = []

    for i, (text, preds, gold_labels) in enumerate(zip(texts, predictions, gold)):
        pred_rows.append({
            "id": i,
            "text": text,
            "gold": gold_labels,
            "predictions": preds,
        })

    predictions_df = pd.DataFrame(pred_rows)

    print(pd.DataFrame([summary]).to_string(index=False))

    print(class_table.to_string(index=False))

    return summary, class_table, predictions_df

def print_email_errors(predictions_df: pd.DataFrame):
    rows = []

    for _, row in predictions_df.iterrows():
        gold = [e for e in row["gold"] if e["label"] == "EMAIL"]
        pred = [e for e in row["predictions"] if e["label"] == "EMAIL"]

        gold_set = {entity_key(e) for e in gold}
        pred_set = {entity_key(e) for e in pred}

        fp = [p for p in pred if entity_key(p) not in gold_set]
        fn = [g for g in gold if entity_key(g) not in pred_set]

        if fp or fn:
            rows.append({
                "id": row["id"],
                "text": row["text"],
                "false_positive": fp,
                "false_negative": fn,
            })

    errors_df = pd.DataFrame(rows)

    print(f"Количество текстов с ошибками: {len(errors_df)}")

    for _, row in errors_df.iterrows():
        print("\n---")
        print(f"ID: {row['id']}")
        print(f"Текст: {row['text']}")

        if row["false_positive"]:
            print("FP:")
            for e in row["false_positive"]:
                print(f'  "{e["text"]}" [{e["start"]}:{e["end"]}] → EMAIL')

        if row["false_negative"]:
            print("FN:")
            for e in row["false_negative"]:
                start, end = e["start"], e["end"]
                print(f'  "{row["text"][start:end]}" [{start}:{end}] → EMAIL')

    return errors_df




In [ ]:
class ContextIdDetector:
    def __init__(self):
        strong_context = (
            r"код\s+заказа|"
            r"код\s+операции|"
            r"код\s+товара|"
            r"внутренний\s+код(?:\s+товара)?|"
            r"внутренний\s+идентификатор|"
            r"идентификатор|"
            r"номер\s+договора|"
            r"номер\s+сч[её]та|"
            r"регистрационный\s+номер"
        )

        self.patterns = [
            rf"(?P<context>{strong_context})\s+(?P<value>\+?\d[\d\s\-.()–—]{{3,30}}\d)",
            rf"(?P<context>{strong_context})\s+(?P<value>[A-ZА-ЯЁ0-9]+(?:[-./][A-ZА-ЯЁ0-9]+)+)",

            r"(?P<context>указал\s+индекс)\s+(?P<value>\d{7,9})",
            r"(?P<context>индекс\s+отправления)\s+(?P<value>\d{7,9})",
            r"(?P<context>указан\s+код)\s+(?P<value>\d{3}\s+\d{3}\s+\d{2}\s+\d{2})",
            r"(?P<context>используется\s+номер)\s+(?P<value>\d{3}-\d{3}-\d{4})(?=\s+как\s+идентификатор)",
        ]

    def predict_one(self, text: str):
        entities = []

        for pattern in self.patterns:
            for m in re.finditer(pattern, text, flags=re.IGNORECASE):
                start = m.start("value")
                end = m.end("value")

                while end > start and text[end - 1] in ".,;:!?)]}":
                    end -= 1

                value = text[start:end]
                digits = re.sub(r"\D", "", value)

                if len(digits) < 4:
                    continue

                if re.search(r"\d+\s*[–—]\s*\d+", value):
                    continue

                entities.append({
                    "start": start,
                    "end": end,
                    "label": "ID",
                    "text": value,
                    "source": "context_id",
                    "source_component": "rule",
                    "source_detector": "context_id",
                })

        return entities

In [ ]:
import ast
import time
import pandas as pd
from navec import Navec
from slovnet import NER as SlovnetNER

TEST_CSV = "/kaggle/input/datasets/shashlovaelizaveta/test-sm/test_small - test_final_300_v3.csv.csv"
NO_PD_CSV = "/kaggle/working/no_pd_eval_clean.csv"
TRICKY_CSV = "/kaggle/input/datasets/shashlovaelizaveta/4-data/tricky_139_final_labeled - tricky_139_final_labeled_converted.csv (7).csv"

NAVEC_PATH = "/kaggle/working/slovnet/scripts/05_ner/navec/navec_news_v1_1B_250K_300d_100q.tar"

PACK_PATH = "/kaggle/working/slovnet/scripts/05_ner/slovnet_ner_pii_ru_hard_no_pd.tar"

ENTITY_CLASSES = ["PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"]
RULE_LABELS = {"PHONE", "EMAIL", "ID"}
ML_LABELS = {"PERSON", "ADDRESS"}

navec = Navec.load(NAVEC_PATH)

ner_model = SlovnetNER.load(PACK_PATH)
ner_model.navec(navec)


def safe_parse_label(x):
    if x is None or pd.isna(x):
        return []

    if isinstance(x, float):
        return []

    if isinstance(x, str):
        x = x.strip()
        if x == "" or x == "[]":
            return []
        return ast.literal_eval(x)

    if isinstance(x, list):
        return x

    return []

def load_dataset(path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(path, sep="\t")
        if "text" not in df.columns:
            df = pd.read_csv(path)
    except Exception:
        df = pd.read_csv(path)

    if "label" not in df.columns:
        df["label"] = [[] for _ in range(len(df))]
    else:
        df["label"] = df["label"].apply(safe_parse_label)

    return df


def normalize_gold(label_list):
    if not isinstance(label_list, list):
        return []

    result = []

    for x in label_list:
        if not isinstance(x, (list, tuple)) or len(x) < 3:
            continue

        result.append({
            "start": int(x[0]),
            "end": int(x[1]),
            "label": str(x[2]),
            "text": None,
            "source": "gold",
            "score": 1.0,
        })

    return result


def entity_key(ent):
    return (
        int(ent["start"]),
        int(ent["end"]),
        str(ent["label"])
    )


def spans_overlap(a, b):
    return not (
        a["end"] <= b["start"]
        or a["start"] >= b["end"]
    )


def ent_len(ent):
    return int(ent["end"]) - int(ent["start"])


def add_text_field(ent, full_text):
    ent = dict(ent)
    ent["start"] = int(ent["start"])
    ent["end"] = int(ent["end"])
    ent["label"] = str(ent["label"])

    if "text" not in ent or ent["text"] is None:
        ent["text"] = full_text[ent["start"]:ent["end"]]

    if "score" not in ent or ent["score"] is None:
        ent["score"] = 1.0

    if "source" not in ent:
        ent["source"] = ent.get("source_component", "unknown")

    return ent



CLASS_PRIORITY = {
    "EMAIL": 5,
    "PHONE": 5,
    "ID": 5,
    "ADDRESS": 4,
    "PERSON": 3,
}


def choose_better_entity(a, b):

    a_label = a["label"]
    b_label = b["label"]

    a_source = a.get("source", "")
    b_source = b.get("source", "")

    a_is_rule_formal = a_label in RULE_LABELS and a_source == "rule"
    b_is_rule_formal = b_label in RULE_LABELS and b_source == "rule"

    if a_is_rule_formal and not b_is_rule_formal:
        return a

    if b_is_rule_formal and not a_is_rule_formal:
        return b

    if a_is_rule_formal and b_is_rule_formal:
        if ent_len(a) != ent_len(b):
            return a if ent_len(a) > ent_len(b) else b
        return a if a.get("score", 1.0) >= b.get("score", 1.0) else b

    if a_label == "ADDRESS" and b_label == "PERSON":
        return a

    if b_label == "ADDRESS" and a_label == "PERSON":
        return b

    if a_label == b_label:
        if ent_len(a) != ent_len(b):
            return a if ent_len(a) > ent_len(b) else b

        if a.get("score", 1.0) != b.get("score", 1.0):
            return a if a.get("score", 1.0) > b.get("score", 1.0) else b

    if a.get("score", 1.0) != b.get("score", 1.0):
        return a if a.get("score", 1.0) > b.get("score", 1.0) else b

    pa = CLASS_PRIORITY.get(a_label, 0)
    pb = CLASS_PRIORITY.get(b_label, 0)

    if pa != pb:
        return a if pa > pb else b

    if ent_len(a) != ent_len(b):
        return a if ent_len(a) > ent_len(b) else b

    return a


def resolve_hybrid_overlaps(entities):
    entities = sorted(
        entities,
        key=lambda e: (e["start"], -(e["end"] - e["start"]))
    )

    selected = []

    for ent in entities:
        current = ent
        keep_current = True

        changed = True
        while changed:
            changed = False

            for old in selected[:]:
                if spans_overlap(current, old):
                    winner = choose_better_entity(current, old)

                    if winner is current:
                        selected.remove(old)
                        changed = True
                        break
                    else:
                        keep_current = False
                        changed = False
                        break

        if keep_current:
            selected.append(current)

    return sorted(selected, key=lambda e: e["start"])


class RuleBasedPIIExtractor:
    def __init__(self):
        self.detectors = {
            "ID_CONTEXT": ContextIdDetector(),
            "PHONE": YargyPhoneDetector(),
            "EMAIL": YargyEmailDetector(),
            "ID": YargyIdDetector(),
        }

    def predict_one(self, text: str):
        entities = []

        for detector_name, detector in self.detectors.items():
            preds = detector.predict_one(text)

            for ent in preds:
                ent = dict(ent)

                if detector_name == "ID_CONTEXT":
                    ent["label"] = "ID"
                else:
                    ent["label"] = detector_name

                ent["start"] = int(ent["start"])
                ent["end"] = int(ent["end"])
                ent["text"] = text[ent["start"]:ent["end"]]
                ent["source"] = "rule"
                ent["source_detector"] = ent.get("source", detector_name)
                ent["score"] = 1.0

                entities.append(ent)

        return resolve_hybrid_overlaps(entities)


class SlovnetPersonAddressExtractor:
    def __init__(self, ner_model):
        self.ner_model = ner_model

    def predict_one(self, text: str):
        markup = self.ner_model(str(text))
        entities = []

        for span in markup.spans:
            label = str(span.type)

            if label in ML_LABELS:
                start = int(span.start)
                end = int(span.stop)

                entities.append({
                    "start": start,
                    "end": end,
                    "label": label,
                    "text": text[start:end],
                    "source": "ml",
                    "source_detector": "slovnet_5class",
                    "score": 1.0,
                })

        return entities


class HybridPIIExtractor:
    def __init__(self, rule_extractor, ml_extractor):
        self.rule_extractor = rule_extractor
        self.ml_extractor = ml_extractor

    def predict_one(self, text: str):
        rule_entities = self.rule_extractor.predict_one(text)
        ml_entities = self.ml_extractor.predict_one(text)

        all_entities = []

        for ent in rule_entities + ml_entities:
            all_entities.append(add_text_field(ent, text))

        return resolve_hybrid_overlaps(all_entities)


rule_extractor = RuleBasedPIIExtractor()
ml_extractor = SlovnetPersonAddressExtractor(ner_model)
hybrid_extractor = HybridPIIExtractor(rule_extractor, ml_extractor)

In [ ]:
def calculate_metrics(gold_all, pred_all, classes=ENTITY_CLASSES):
    per_class = {}
    total_tp = total_fp = total_fn = 0

    for cls in classes:
        tp = fp = fn = 0

        for gold, pred in zip(gold_all, pred_all):
            gold_cls = {entity_key(e) for e in gold if e["label"] == cls}
            pred_cls = {entity_key(e) for e in pred if e["label"] == cls}

            tp += len(gold_cls & pred_cls)
            fp += len(pred_cls - gold_cls)
            fn += len(gold_cls - pred_cls)

        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall
            else 0.0
        )

        per_class[cls] = {
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "TP": tp,
            "FP": fp,
            "FN": fn,
        }

        total_tp += tp
        total_fp += fp
        total_fn += fn

    micro_precision = total_tp / (total_tp + total_fp) if total_tp + total_fp else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if total_tp + total_fn else 0.0

    micro_f1 = (
        2 * micro_precision * micro_recall / (micro_precision + micro_recall)
        if micro_precision + micro_recall
        else 0.0
    )

    macro_f1 = sum(per_class[c]["F1"] for c in classes) / len(classes)

    return {
        "per_class": per_class,
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
    }


def metrics_to_table(metrics):
    rows = []

    for cls in ENTITY_CLASSES:
        m = metrics["per_class"][cls]
        rows.append({
            "Класс": cls,
            "Precision": round(m["Precision"], 4),
            "Recall": round(m["Recall"], 4),
            "F1": round(m["F1"], 4),
            "TP": m["TP"],
            "FP": m["FP"],
            "FN": m["FN"],
        })

    return pd.DataFrame(rows)


def run_hybrid_experiment(path: str, name: str = "DATASET"):
    df = load_dataset(path)

    texts = df["text"].astype(str).tolist()
    gold = [normalize_gold(x) for x in df["label"].tolist()]

    for text, ents in zip(texts, gold):
        for ent in ents:
            ent["text"] = text[ent["start"]:ent["end"]]

    start_time = time.time()
    predictions = [
        hybrid_extractor.predict_one(text)
        for text in texts
    ]
    elapsed = time.time() - start_time

    metrics = calculate_metrics(gold, predictions)
    class_table = metrics_to_table(metrics)

    summary = {
        "dataset": name,
        "method": "Hybrid: rules(PHONE/EMAIL/ID) + Slovnet5(PERSON/ADDRESS)",
        "micro_precision": round(metrics["micro_precision"], 4),
        "micro_recall": round(metrics["micro_recall"], 4),
        "micro_f1": round(metrics["micro_f1"], 4),
        "macro_f1": round(metrics["macro_f1"], 4),
        "time_sec": round(elapsed, 4),
        "texts_per_sec": round(len(df) / elapsed, 4) if elapsed > 0 else None,
    }

    pred_rows = []

    for i, (text, preds, gold_labels) in enumerate(zip(texts, predictions, gold)):
        pred_rows.append({
            "id": i,
            "text": text,
            "gold": gold_labels,
            "predictions": preds,
        })

    predictions_df = pd.DataFrame(pred_rows)

    print(pd.DataFrame([summary]).to_string(index=False))

    display(class_table)

    return summary, class_table, predictions_df


test_summary, test_table, test_predictions = run_hybrid_experiment(TEST_CSV, "TEST")


Ошибки

In [ ]:
import pandas as pd

ENTITY_CLASSES = ["PERSON", "PHONE", "EMAIL", "ADDRESS", "ID"]


def ent_value(text, ent):
    return text[int(ent["start"]):int(ent["end"])]


def collect_errors(df, dataset_name):
    rows = []

    for idx, row in df.iterrows():
        text = str(row["text"])
        source = row["source"] if "source" in df.columns else dataset_name

        gold = normalize_gold(row["label"])
        pred = hybrid_extractor.predict_one(text)

        for ent in gold:
            ent["text"] = ent_value(text, ent)

        for ent in pred:
            ent["text"] = ent_value(text, ent)

        gold_set = {entity_key(e) for e in gold}
        pred_set = {entity_key(e) for e in pred}

        for ent in pred:
            if entity_key(ent) not in gold_set:
                rows.append({
                    "dataset": dataset_name,
                    "source": source,
                    "text_id": idx,
                    "class": ent["label"],
                    "error_type": "FP",
                    "text": text,
                    "gold": "",
                    "predicted": ent["text"],
                    "pred_start": ent["start"],
                    "pred_end": ent["end"],
                    "gold_start": "",
                    "gold_end": "",
                    "pred_source": ent.get("source", ""),
                    "pred_detector": ent.get("source_detector", ""),
                })

        for ent in gold:
            if entity_key(ent) not in pred_set:
                rows.append({
                    "dataset": dataset_name,
                    "source": source,
                    "text_id": idx,
                    "class": ent["label"],
                    "error_type": "FN",
                    "text": text,
                    "gold": ent["text"],
                    "predicted": "",
                    "pred_start": "",
                    "pred_end": "",
                    "gold_start": ent["start"],
                    "gold_end": ent["end"],
                    "pred_source": "",
                    "pred_detector": "",
                })

    return pd.DataFrame(rows)


def no_pd_summary(no_pd_errors_df, no_pd_df):
    n_texts = len(no_pd_df)
    texts_with_error = no_pd_errors_df["text_id"].nunique()
    fp_entities = len(no_pd_errors_df[no_pd_errors_df["error_type"] == "FP"])
    fp_text_rate = texts_with_error / n_texts if n_texts else 0.0

    return {
        "n_texts": n_texts,
        "texts_with_error": texts_with_error,
        "fp_entities": fp_entities,
        "fp_text_rate": round(fp_text_rate, 4),
    }


def tricky_summary(tricky_errors_df, tricky_df):
    n_texts = len(tricky_df)
    texts_with_any_error = tricky_errors_df["text_id"].nunique()
    fp = len(tricky_errors_df[tricky_errors_df["error_type"] == "FP"])
    fn = len(tricky_errors_df[tricky_errors_df["error_type"] == "FN"])
    error_text_rate = texts_with_any_error / n_texts if n_texts else 0.0

    return {
        "n_texts": n_texts,
        "texts_with_any_error": texts_with_any_error,
        "FP": fp,
        "FN": fn,
        "error_text_rate": round(error_text_rate, 4),
    }



In [ ]:
test_df = load_dataset(TEST_CSV)
no_pd_df = load_dataset(NO_PD_CSV)
tricky_df = load_dataset(TRICKY_CSV)


test_errors_df = collect_errors(test_df, "TEST")
no_pd_errors_df = collect_errors(no_pd_df, "NO-PD")
tricky_errors_df = collect_errors(tricky_df, "TRICKY")

In [ ]:
no_pd_stats = no_pd_summary(no_pd_errors_df, no_pd_df)

for k, v in no_pd_stats.items():
    print(f"{k}: {v}")

In [ ]:
tricky_stats = tricky_summary(tricky_errors_df, tricky_df)

for k, v in tricky_stats.items():
    print(f"{k}: {v}")

In [ ]:
display(
    test_errors_df
    .groupby(["class", "error_type"])
    .size()
    .reset_index(name="count")
    .sort_values(["class", "error_type"])
)

display(test_errors_df)

In [ ]:
display(tricky_errors_df)

In [ ]:
display(no_pd_errors_df)


In [ ]:
test_errors_df.to_csv("/kaggle/working/hybrid_test_errors.csv", index=False)
tricky_errors_df.to_csv("/kaggle/working/hybrid_tricky_errors.csv", index=False)
no_pd_errors_df.to_csv("/kaggle/working/hybrid_no_pd_errors.csv", index=False)

print("/kaggle/working/hybrid_test_errors.csv")
print("/kaggle/working/hybrid_tricky_errors.csv")
print("/kaggle/working/hybrid_no_pd_errors.csv")

In [ ]:
!ls /kaggle/working/slovnet/scripts/05_ner/*.tar